## tl;dr

将 `SUBMISSION_LINK_AUTO_THRESHOLD` 从 **0.85** 下调到 **0.65**，保留标题分 `>= 0.70` 的既有保护门槛；`SUBMISSION_LINK_REVIEW_THRESHOLD` 继续保持 **0.55**。在 2026-08-14 当前链接算法上线后的 582 条记录中，新阈值会把原本 104 条待确认中的 94 条自动确认，待确认量预计减少 90.4%。历史人工标签中，满足新自动确认条件的 127 条全部被确认；4 条否决的综合分均低于 0.63。

## Context & Methods

### Key Assumptions

- 分析单元是 `submitted_report_items` 的一条存档条目。
- `link_decided_by is not null` 视为人工触达；`rejected` 视为否决，`matched` 且 `article_id = best_candidate_article_id` 视为人工确认最佳候选。
- 2026-08-14 16:32:58+08:00 之后使用当前候选选择与阈值逻辑；更早记录存在“标题完全相同直接通过”的旧逻辑，因此只用于补充人工标签，不用于估算当前队列缩减。
- 表只保存最终状态而非完整决策事件流，结论适合校准阈值，但不能当作长期线上误匹配率的无偏估计。

In [ ]:
from src.adapters.db_postgres_core import get_adapter

adapter = get_adapter()
cursor = adapter._conn_cursor()
cursor.execute("""
select count(*) as total, min(created_at) as min_created,
       max(created_at) as max_created,
       count(*) filter (where link_decided_by is not null) as human_touched
from submitted_report_items
""")
dict(cursor.fetchone())


## Data

查询时共有 1,406 条记录、132 份报告，覆盖 2026-07-29 至 2026-09-05；176 条记录有人工作用。质量检查未发现主键重复、分数缺失/越界、非法状态、孤立候选、回链状态与 `article_id` 不一致或报告条目数不一致。

In [ ]:
cursor.execute("""
with thresholds(t) as (
    values (0.60), (0.62), (0.63), (0.64), (0.65), (0.70), (0.75), (0.80)
), labeled as (
    select i.*, case
        when link_status = 'rejected' then false
        when link_status = 'matched'
          and article_id = best_candidate_article_id then true
        else null end as accepted
    from submitted_report_items i
    where link_decided_by is not null
)
select t,
       count(*) filter (where link_combined_score >= t
         and link_title_score >= 0.70 and accepted is not null) as labeled_n,
       count(*) filter (where link_combined_score >= t
         and link_title_score >= 0.70 and accepted) as accepted_n,
       count(*) filter (where link_combined_score >= t
         and link_title_score >= 0.70 and accepted = false) as rejected_n
from thresholds cross join labeled group by t order by t
""")
[dict(row) for row in cursor.fetchall()]


## Results

| 自动阈值 | 有人工标签的候选 | 确认 | 否决 |
|---:|---:|---:|---:|
| 0.60 | 141 | 140 | 1 |
| 0.62 | 137 | 136 | 1 |
| 0.63 | 134 | 134 | 0 |
| 0.64 | 130 | 130 | 0 |
| **0.65** | **127** | **127** | **0** |
| 0.70 | 107 | 107 | 0 |
| 0.75 | 75 | 75 | 0 |
| 0.80 | 45 | 45 | 0 |

0.63 是样本内零否决的最低分界，但它只比最高否决分 0.6256 高 0.0044。选择 0.65 留出更稳妥的缓冲，同时在当前算法时期仍能把 94/104 条待确认自动化。

## Takeaways

- 采用自动阈值 0.65，并继续要求标题分至少 0.70。
- 人工复核下限 0.55 不下调：4 条否决全部集中在 0.5515 至 0.6256，说明这段仍需要人看。
- 上线后应继续记录 0.65 至 0.70 区间的人工纠错；由于目前缺少不可变决策事件日志，建议以人工解绑/改绑作为误匹配监控信号。